In [1]:
from rlrom.trainers import RLTrainer
from rlrom.testers import RLTester
import os, sys, glob
from rlrom.utils import *
from pprint import pprint
import polars as pl
from datetime import date
import ipywidgets as widgets
from IPython.display import display
wd = os.getcwd()
print(wd) 

I0000 00:00:1790862119.132535   63030 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790862119.159976   63030 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790862119.871522   63030 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


/local/home/donzeal/workspace/rlrom/examples


In [2]:
w = widgets.Dropdown(
    options=[
        'cartpole/cfg0tr_ppo_specs.yml',
        'cartpole/cfg0tr_ppo.yml',
        'highway_env/cfg0tr.yml'])
display(w)


Dropdown(options=('cartpole/cfg0tr_ppo_specs.yml', 'cartpole/cfg0tr_ppo.yml', 'highway_env/cfg0tr.yml'), value…

In [3]:
os.chdir(wd)
cfg_file = w.value
print(cfg_file)
cfg = load_cfg(cfg_file)
T = RLTester(cfg)

cartpole/cfg0tr_ppo_specs.yml
loading field [ specs ] from STL file [ cartpole.stl ]
loading field [ ppo ] from YAML file [ cartpole_ppo.yml ]


In [5]:
df_train_files = get_df_all_training_files(cfg)
df_train_res = get_df_all_training_res(df_train_files)
print(df_train_res)


shape: (318, 10)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ steps  ┆ res_files  ┆ model_file ┆ path      ┆ … ┆ mean_ep_r ┆ eval_form ┆ reward_fo ┆ label     │
│ ---    ┆ ---        ┆ s          ┆ ---       ┆   ┆ ew        ┆ ulas      ┆ rmulas    ┆ ---       │
│ i64    ┆ str        ┆ ---        ┆ str       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ str       │
│        ┆            ┆ str        ┆           ┆   ┆ f64       ┆ struct[2] ┆ struct[1] ┆           │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 10000  ┆ /local/hom ┆ model_step ┆ /local/ho ┆ … ┆ 38.248376 ┆ {{-0.0423 ┆ {{3.5,-0. ┆ Training0 │
│        ┆ e/donzeal/ ┆ _10000.zip ┆ me/donzea ┆   ┆           ┆ 07,3.5,-1 ┆ 042307,-1 ┆           │
│        ┆ workspace/ ┆            ┆ l/workspa ┆   ┆           ┆ .189817}, ┆ .189817}} ┆           │
│        ┆ …          ┆            ┆ ce/…      ┆   ┆           ┆ {-0…     

In [ ]:
df_all_training_res = None
safe_select = ['label', 'steps','mean_ep_rew', 'mean_ep_len', 'res', 'res_files', 'model_files', 'path']
idx =0
for r in df_all_trainings['training_files']:    
    r = get_df_load_training_res(r,f'Training{idx}')
    #r = r.select(safe_select)
    if df_all_training_res is None:
        df_all_training_res = r
    else:
        df_all_training_res = pl.concat([   df_all_training_res,
                                            r])
    idx = idx+1


In [ ]:
print(r)

In [ ]:

dfallt= get_df_all_training_files(cfg).collect()
r =  dfallt['training_files'][0]
idx=0
r = get_df_load_training_res(r,f'Training{idx}')
safe_select = ['label', 'steps','mean_ep_len', 'mean_ep_rew', 'res', 'res_files', 'model_files', 'path', 'mean_ep_len', 'mean_ep_rew']

print(r.collect().select([ col for col in r.collect().columns if col not in safe_select ]))


In [ ]:
safe_select = ['steps', 'res_files', 'model_files', 'path']
print(r.collect().select(['steps', 'res_files', 'model_files', 'path']))

In [ ]:
# plot training results
import rlrom.plots as rlp

df_all_training_files = get_df_all_training_files(cfg)

df = get_df_training(cfg, train_idx)
rlp.plot_df_training(df)



In [ ]:
# Testing "Best" model
out = get_best_models(cfg, train_idx=4)
m = out[0]
#m = get_step_model(cfg,100000)
set_active_model(cfg, out[0])
show_active_model(cfg)
res = T.run_cfg_test()
pprint(res['res_all_ep'])

In [ ]:
pprint(res['res_all_ep'])
#pprint(res['res']['cart_on_left'])


In [ ]:
T.run_episode(seed=4)
d = T.env.get_wrapper_attr('stl_driver')
x = d.get_signal('x')
print(x)
ax = d.plot_signal('x')
d.plot_rob('cart_on_left', 0,90, ax=ax)


In [ ]:
print(d)